In [3]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
llm = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")

In [6]:
with open("blood_work.txt", "r") as f:
    blood_report =f.read()
print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5â€“17.5)
Hematocrit:        44%          


In [8]:
extraction_prompt = f"""You are a clinical lab report parser. Extract every test result from the blood report below.

For each test, return exactly one line in this format:
Test Name | Result | Unit | Status | Reference Range

Rules:
- Status must be one of: HIGH, LOW, NORMAL
- Compare the result against the reference range provided in the report to determine status
- If a reference range is missing, set status to UNKNOWN
- Do not skip any test, even if the value looks unremarkable
- Do not add any commentary, headers, or explanations
- Use the exact test name as written in the report

Blood Report:
{blood_report}
"""
extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("======= STAGE 1: Extracted Values ====")
print(extracted_values)

======= STAGE 1: Extracted Values ====
Hemoglobin | 15.1 | g/dL | NORMAL | 13.5-17.5
Hematocrit | 44 | % | NORMAL | 41-53%
WBC | 6.8 | x10^3/uL | NORMAL | 4.5-11.0
Platelets | 220 | x10^3/uL | NORMAL | 150-400
Total Cholesterol | 238 | mg/dL | HIGH | <200
LDL Cholesterol | 162 | mg/dL | HIGH | <100
HDL Cholesterol | 36 | mg/dL | LOW | >40
Triglycerides | 188 | mg/dL | HIGH | <150
Glucose (Fasting) | 92 | mg/dL | NORMAL | 70-99
HbA1c | 5.3 | % | NORMAL | <5.7%
Creatinine | 1.0 | mg/dL | NORMAL | 0.7-1.3
eGFR | 82 | mL/min | NORMAL | >60
ALT | 28 | U/L | NORMAL | 7-40
AST | 25 | U/L | NORMAL | 10-40
Bilirubin Total | 0.8 | mg/dL | NORMAL | 0.2-1.2


In [10]:
diet_prompt = f"""You are an Indian clinical nutritionist. Given this blood work, respond in exactly two sections:

## Health Summary
4-5 lines, simple language, what's off and what it means.

## Diet Plan
**Eat more:** 8-10 specific Indian foods (palak, ragi, chana — not "green vegetables"). Note which blood value each helps.
**Avoid:** 5-7 specific items. Note which value each worsens.

Food only, no supplements. Prioritize affordable, common ingredients.

Blood Work:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)



In [11]:
print("====== STAGE 2: Health Summary & Diet Plan")
print(diet_response.text)

====== STAGE 2: Health Summary & Diet Plan
## Health Summary

Your blood test shows excellent results for your blood sugar, kidney function, liver health, and blood counts. However, your cholesterol panel needs attention, as your total cholesterol, bad cholesterol (LDL), and triglycerides are high, while your good cholesterol (HDL) is low. In daily terms, this means extra fats are floating in your bloodstream, which can slowly build up in your blood vessels and strain your heart over time. The overall picture is very positive because your core organ health is strong, and cholesterol levels respond quickly to routine diet changes. By eating more fiber-rich traditional foods and switching to lower-fat cooking methods, you can successfully bring your cholesterol back into a healthy range.

## Diet Plan

**Foods to eat more of:**
- **Ragi (Finger Millet):** Helps lower **LDL and Triglycerides**. It is rich in soluble fiber that binds to cholesterol in your gut and flushes it out before it 